In [ ]:

import glob
import os
import sys
import io
import pytesseract
from PIL import Image
import textwrap
from PyPDF2 import PdfReader
from nltk.tokenize import sent_tokenize
import nltk
import pickle
import warnings
from tqdm import tqdm
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings
from langchain_openai import ChatOpenAI
from langchain.chains import RetrievalQA
import pickle as pkl
import fitz  # PyMuPDF
import docx
import docx2txt
warnings.filterwarnings("ignore", category=UserWarning)
nltk.download('punkt')

/Users/piyush/workspace/projects/DeepLearning/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
[nltk_data] Downloading package punkt to /Users/piyush/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [ ]:
# Here is your openai key.
#Note: if this key stops working (very likely as many people are using it), you should get your own API Key and replace the old key.

#################
openai_key = "your_api_key" ## Put your Open_AI Key here
#################
os.environ["OPENAI_API_KEY"] = openai_key


-----


In [5]:
import os
import glob
import tiktoken
import docx2txt
import fitz  # PyMuPDF
import pytesseract
from PIL import Image
from tqdm import tqdm
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

# Set file path
filepaths = glob.glob("/Users/piyush/workspace/projects/DeepLearning/Chatbot-AIEB/10k/*.*")

# Extract text from different file types
def extract_text(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext in ['.pdf']:
        return "\n".join([page.get_text() for page in fitz.open(filepath)])
    elif ext in ['.doc', '.docx']:
        return docx2txt.process(filepath)
    elif ext in ['.png', '.jpg', '.jpeg']:
        return pytesseract.image_to_string(Image.open(filepath))
    return ""

# Token-Based Chunking (OpenAI Tokenizer)
def token_chunk(text, model="text-embedding-3-large", size=600, overlap=100):
    encoding = tiktoken.encoding_for_model(model)
    tokens = encoding.encode(text)
    return [encoding.decode(tokens[i:i + size]) for i in range(0, len(tokens), size - overlap)]

# Process files
chunked_dataset = {}
for i, filepath in enumerate(tqdm(filepaths, desc="Processing Files")):
    text = extract_text(filepath)
    if text:
        chunked_dataset[i] = token_chunk(text)
        
# Store in FAISS
embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
vectorstores = []
for doc_num, chunks in tqdm(chunked_dataset.items(), desc="Storing in FAISS"):
    store = FAISS.from_texts(chunks, embedding=embeddings)
    vectorstores.append(store)

# Merge all vector stores
vectorstore = vectorstores[0]
for store in vectorstores[1:]:
    vectorstore.merge_from(store)

# Save vector database
vectorstore.save_local(folder_path="./Vectorstores", index_name="Resumes")
print("FAISS Vectorstore saved successfully!")

Processing Files: 100%|██████████| 3/3 [00:01<00:00,  2.10it/s]
/var/folders/4w/zz44xt0x4yx19mn8x1c1qzcr0000gn/T/ipykernel_20745/1404114361.py:40: LangChainDeprecationWarning: The class `OpenAIEmbeddings` was deprecated in LangChain 0.0.9 and will be removed in 1.0. An updated version of the class exists in the :class:`~langchain-openai package and should be used instead. To use it run `pip install -U :class:`~langchain-openai` and import as `from :class:`~langchain_openai import OpenAIEmbeddings``.
  embeddings = OpenAIEmbeddings(model="text-embedding-3-large")
Storing in FAISS: 100%|██████████| 3/3 [00:07<00:00,  2.64s/it]

FAISS Vectorstore saved successfully!


# Ask llm questions



In [ ]:
top_k_docs = 10

llm = ChatOpenAI(model= "gpt-4", temperature=0.7, max_retries = 10) ### Change it as you wish

embeddings = OpenAIEmbeddings(model="text-embedding-3-large")

vectorstore = FAISS.load_local(folder_path=f"./Vectorstores",
                                  embeddings=embeddings,
                                  index_name="Resumes",
                                  allow_dangerous_deserialization=True)


retriever = vectorstore.as_retriever(search_type = "similarity", search_kwargs={"k":top_k_docs})
qa = RetrievalQA.from_chain_type(
        llm = llm,
        chain_type = "stuff",
        retriever = retriever,
        verbose = True,
        return_source_documents = True
    )



In [ ]:


query_history = ""

while True:
    query = input('Enter your query: (type "exit" to exit) \n')

    if query.strip().lower() == "exit":
        # print("Answer from AI HR manager: \n")
        print("goodbye")
        break

    print("\n \033[1mloading... \033[0m\n")

    original_stdout = sys.stdout
    captured_output = io.StringIO()
    sys.stdout = captured_output

    sources = vectorstore.similarity_search_with_relevance_scores(query, k=5)

    result = qa.invoke({
        # "persona": persona,
        "query": "<history>" +query_history+"</history> \n <query>"+query+"</query>",
    })

    sys.stdout = original_stdout

    output = captured_output.getvalue()
    filtered_output = output.replace("Entering new RetrievalQA chain...\n", "").replace("Finished chain.\n", "").replace("loading...\n", "")


    query_history += f"User: {query}\n"
    query_history += f"Bot: {result['result']}\n"

    formatted_response = textwrap.dedent(result['result'])
    wrapped_string = textwrap.fill(formatted_response, width=60)

    print(wrapped_string)
    print("\n\n")


 loading... 

Answer from AI HR manager:
Amazon's total revenue for Q3 2024 was $158,877 million.




 loading... 

Answer from AI HR manager:
Microsoft's AI strategy focuses on developing and
incorporating AI into their existing and new products and
services, while ensuring a responsible design and build with
safety and security from the outset. They believe AI should
be as empowering across communities as it is powerful. They
aim to create platforms and tools, powered by AI, that offer
innovative solutions to meet evolving customer needs. They
also focus on AI-optimized infrastructure, to provide
customers with a variety of large language models and
developer tools to create the next generation of AI apps and
services. Microsoft's AI offerings, including Copilot and
their Copilot stack, are aimed at orchestrating a new era of
AI transformation, driving better business outcomes across
every role and industry.




 loading... 

Answer from AI HR manager:
The document mentions several 